# 04 · ข้อ 3 ผู้โดยสารระหว่างประเทศ ขาเข้า vs ขาออก
ผู้รับผิดชอบ: สมาชิก C · ต้องรัน `01_cleaning.ipynb` (หรือ `python src/run_all.py`) ก่อน

**คำถาม:** จำนวนผู้โดยสารระหว่างประเทศขาเข้าและขาออกมีความแตกต่างกันอย่างไร เมื่อจำแนกตามรูปแบบการเดินทาง เปรียบเทียบรายเดือน และวิเคราะห์ความผันผวนภายในปี

โค้ดทั้งหมดอยู่ใน `src/q3_intl.py` notebook นี้มีหน้าที่**เรียกฟังก์ชันและแสดงผล**พร้อมคำอธิบาย
ฟังก์ชันที่ยังเป็น TODO ให้เขียนใน `src/q3_intl.py` → ตรวจด้วย `pytest tests/test_q3_intl.py` → ลบ `#` หน้า cell ที่เกี่ยวข้อง

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import prep
import features
import plots

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
_ = plots.setup_style()
import q3_intl

df = prep.load_clean()
print(df.shape)

(18705, 18)


## 1. Feature Selection

| Feature ต้นฉบับ | เหตุผล |
|---|---|
| `purpose` | เลือกเฉพาะ international |
| `mode` | จำแนกตามรูปแบบการเดินทาง (ถนน/น้ำ/ราง/อากาศ) |
| `port`, `direction` | จับคู่ขาเข้า–ขาออกของท่าเดียวกัน |
| `date`, `value` | รวมเป็นรายเดือน |

## 2. Data Preparation

In [2]:
rows = q3_intl.select_rows(df)   # error ถ้ามีท่าที่ไม่มีครบทั้งขาเข้าและขาออก
display(rows.groupby(["mode", "port"])["direction"].unique())
print("เดือนที่ข้อมูลไม่ครบ (ต้องใช้ avg_daily แทน total):")
q3_intl.coverage_table(rows)

mode   port                      
air    ท่าอากาศยานดอนเมือง           [departure, arrival]
       ท่าอากาศยานภูมิภาค            [departure, arrival]
       ท่าอากาศยานสุวรรณภูมิ         [departure, arrival]
       ท่าอากาศยานอื่น ๆ ของ ทอท.    [departure, arrival]
rail   รถไฟ                          [departure, arrival]
road   รถ บขส.                       [departure, arrival]
water  ท่าเรือด่านชายแดน             [departure, arrival]
Name: direction, dtype: object

เดือนที่ข้อมูลไม่ครบ (ต้องใช้ avg_daily แทน total):


,year_be,mode,port,direction,year_month,days,valid_days,total,avg_daily,coverage
121,2568,road,รถ บขส.,arrival,2025-02,28,27,17089.0,632.925926,0.964286
122,2568,road,รถ บขส.,arrival,2025-03,31,30,16561.0,552.033333,0.967742
123,2568,road,รถ บขส.,arrival,2025-04,30,29,22787.0,785.758621,0.966667
129,2568,road,รถ บขส.,arrival,2025-10,31,30,14488.0,482.933333,0.967742
130,2568,road,รถ บขส.,arrival,2025-11,30,29,14891.0,513.482759,0.966667
131,2568,road,รถ บขส.,arrival,2025-12,31,30,16273.0,542.433333,0.967742
133,2568,road,รถ บขส.,departure,2025-02,28,27,41428.0,1534.370370,0.964286
134,2568,road,รถ บขส.,departure,2025-03,31,30,39035.0,1301.166667,0.967742
135,2568,road,รถ บขส.,departure,2025-04,30,29,47933.0,1652.862069,0.966667
137,2568,road,รถ บขส.,departure,2025-06,30,29,29704.0,1024.275862,0.966667


## 3. Feature Engineering

| Feature ใหม่ | สูตร / เงื่อนไข | ฟังก์ชัน |
|---|---|---|
| `arrival`, `departure` รายเดือน | รวมรายเดือนต่อ mode ด้วย `features.monthly_from_daily` ✅ | `q3_intl.monthly_flows` (TODO) |
| `net_flow` | `arrival − departure` | `q3_intl.add_flow_features` (TODO) |
| `arr_dep_ratio` | `arrival / departure` | `q3_intl.add_flow_features` (TODO) |
| `mom_pct_*` | % เปลี่ยนจากเดือนก่อน | `q3_intl.add_flow_features` (TODO) |
| `mode_share` | สัดส่วนของ mode ต่อยอดระหว่างประเทศของเดือน | `q3_intl.add_flow_features` (TODO) |
| `cv_within_year` | `std / mean` ของยอดรายเดือน | `q3_intl.volatility_table` (TODO) |

ทางอากาศใหญ่กว่า mode อื่นหลายสิบเท่า → ใช้ ratio/share หรือแยก panel ไม่ใช้แกนเดียว

In [3]:
# monthly = q3_intl.add_flow_features(q3_intl.monthly_flows(rows))
# vol = q3_intl.volatility_table(monthly)
# display(monthly.head(), vol)

## 4. Visualization
กราฟที่วางไว้: ① `plot_arrival_departure` ② `plot_volatility`

In [4]:
# fig = q3_intl.plot_arrival_departure(monthly); plots.save_fig(fig, "q3_arrival_departure"); plt.show()
# fig = q3_intl.plot_volatility(vol); plots.save_fig(fig, "q3_volatility"); plt.show()

**การตีความ:** _(เขียนจากตัวเลขจริงในกราฟ)_

## 5. Analysis & Findings
- คำตอบของคำถาม:
- ข้อค้นพบสำคัญ:
- ข้อจำกัดของข้อมูล:

## 6. บันทึก Prepared Dataset

In [5]:
# monthly.to_csv(prep.PREPARED_DIR / q3_intl.OUTPUT_FILE, index=False, encoding="utf-8-sig")